In [2]:
import sqlite3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. DATABASE INITIALIZATION (CLOUD STORAGE)
# ==========================================
def init_db():
    conn = sqlite3.connect('hostel_cloud_db.db')
    cursor = conn.cursor()

    # Create Rooms Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS rooms (
            room_no TEXT PRIMARY KEY,
            room_type TEXT,
            capacity INTEGER,
            allocated INTEGER DEFAULT 0,
            status TEXT DEFAULT 'Available'
        )
    ''')

    # Create Students Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS students (
            student_id TEXT PRIMARY KEY,
            name TEXT,
            gender TEXT,
            phone TEXT,
            room_no TEXT,
            FOREIGN KEY(room_no) REFERENCES rooms(room_no)
        )
    ''')

    # Seed default room data if empty
    cursor.execute("SELECT COUNT(*) FROM rooms")
    if cursor.fetchone()[0] == 0:
        default_rooms = [
            ('101', 'Single AC', 1, 0, 'Available'),
            ('102', 'Double Non-AC', 2, 0, 'Available'),
            ('103', 'Double AC', 2, 0, 'Available'),
            ('201', 'Triple Non-AC', 3, 0, 'Available'),
            ('202', 'Single Non-AC', 1, 0, 'Available')
        ]
        cursor.executemany("INSERT INTO rooms VALUES (?, ?, ?, ?, ?)", default_rooms)
        conn.commit()
    conn.close()

init_db()

# ==========================================
# 2. CORE SYSTEM FUNCTIONS
# ==========================================
def execute_query(query, params=(), fetch=False):
    conn = sqlite3.connect('hostel_cloud_db.db')
    cursor = conn.cursor()
    cursor.execute(query, params)
    data = None
    if fetch:
        data = cursor.fetchall()
    conn.commit()
    conn.close()
    return data

def allocate_room(student_id, name, gender, phone, room_no):
    # Check if student already exists
    exists = execute_query("SELECT student_id FROM students WHERE student_id=?", (student_id,), fetch=True)
    if exists:
        return "❌ Error: Student ID already exists in the cloud database."

    # Check room capacity status
    room = execute_query("SELECT capacity, allocated FROM rooms WHERE room_no=?", (room_no,), fetch=True)
    if not room:
        return "❌ Error: Room number does not exist."

    capacity, allocated = room[0]
    if allocated >= capacity:
        return "❌ Error: Selected room is already full."

    # Allocate student and increment room count
    execute_query("INSERT INTO students VALUES (?, ?, ?, ?, ?)", (student_id, name, gender, phone, room_no))
    new_allocated = allocated + 1
    status = 'Full' if new_allocated == capacity else 'Available'
    execute_query("UPDATE rooms SET allocated=?, status=? WHERE room_no=?", (new_allocated, status, room_no))

    return f"✅ Success: Student {name} allocated to Room {room_no} successfully!"

def vacate_student(student_id):
    student = execute_query("SELECT name, room_no FROM students WHERE student_id=?", (student_id,), fetch=True)
    if not student:
        return "❌ Error: Student ID not found."

    name, room_no = student[0]

    # Delete student record
    execute_query("DELETE FROM students WHERE student_id=?", (student_id,))

    # Decrement allocated counter for the room
    room = execute_query("SELECT allocated FROM rooms WHERE room_no=?", (room_no,), fetch=True)
    if room:
        allocated = room[0][0]
        new_allocated = max(0, allocated - 1)
        execute_query("UPDATE rooms SET allocated=?, status='Available' WHERE room_no=?", (new_allocated, room_no))

    return f"⚠️ Student {name} vacated successfully from Room {room_no}."

# ==========================================
# 3. INTERACTIVE GOOGLE COLAB UI LAYER
# ==========================================
output_area = widgets.Output()

# UI Styling
style = HTML("<style>.widget-label { font-weight: bold; }</style>")
display(style)

# Navigation Tabs
tab = widgets.Tab()

# --- Tab 1: Allocate Student Layout ---
t1_id = widgets.Text(description="Student ID:")
t1_name = widgets.Text(description="Full Name:")
t1_gender = widgets.Dropdown(options=['Male', 'Female', 'Other'], description="Gender:")
t1_phone = widgets.Text(description="Phone No:")
# Dynamically fetch available rooms
room_choices = [r[0] for r in execute_query("SELECT room_no FROM rooms WHERE status='Available'", fetch=True)]
t1_room = widgets.Dropdown(options=room_choices, description="Room No:")
t1_btn = widgets.Button(description="Allocate Room", button_style='success', icon='check')

t1_form = widgets.VBox([t1_id, t1_name, t1_gender, t1_phone, t1_room, t1_btn])

# --- Tab 2: Vacate Student Layout ---
t2_id = widgets.Text(description="Student ID:")
t2_btn = widgets.Button(description="Vacate Student", button_style='danger', icon='trash')
t2_form = widgets.VBox([t2_id, t2_btn])

# --- Tab 3: Cloud Dashboards Layout ---
t3_btn_students = widgets.Button(description="View All Students", button_style='info', icon='users')
t3_btn_rooms = widgets.Button(description="View Room Status", button_style='info', icon='home')
t3_actions = widgets.HBox([t3_btn_students, t3_btn_rooms])
t3_form = widgets.VBox([t3_actions])

# Assemble Dashboard Tabs
tab.children = [t1_form, t2_form, t3_form]
tab.set_title(0, 'Add Allocation')
tab.set_title(1, 'Vacate Room')
tab.set_title(2, 'Cloud Dashboard')

# ==========================================
# 4. INTERACTION LOGIC & EVENT HANDLERS
# ==========================================
def refresh_room_dropdowns():
    rooms = [r[0] for r in execute_query("SELECT room_no FROM rooms WHERE status='Available'", fetch=True)]
    t1_room.options = rooms

def on_allocate_clicked(b):
    with output_area:
        clear_output()
        if not (t1_id.value and t1_name.value and t1_phone.value and t1_room.value):
            print("⚠️ Please fill out all fields completely.")
            return
        msg = allocate_room(t1_id.value, t1_name.value, t1_gender.value, t1_phone.value, t1_room.value)
        print(msg)
        refresh_room_dropdowns()

def on_vacate_clicked(b):
    with output_area:
        clear_output()
        if not t2_id.value:
            print("⚠️ Please enter a valid Student ID.")
            return
        msg = vacate_student(t2_id.value)
        print(msg)
        refresh_room_dropdowns()

def on_view_students_clicked(b):
    with output_area:
        clear_output()
        conn = sqlite3.connect('hostel_cloud_db.db')
        df = pd.read_sql_query("SELECT * FROM students", conn)
        conn.close()
        if df.empty:
            print("📂 Cloud Database Notice: No students are currently registered in the hostel.")
        else:
            display(HTML("<h3>📋 Registered Students Inventory</h3>"))
            display(df)

def on_view_rooms_clicked(b):
    with output_area:
        clear_output()
        conn = sqlite3.connect('hostel_cloud_db.db')
        df = pd.read_sql_query("SELECT * FROM rooms", conn)
        conn.close()
        display(HTML("<h3>🏢 Real-time Room Allocation Tracker</h3>"))
        display(df)

# Connect Buttons to Logic
t1_btn.on_click(on_allocate_clicked)
t2_btn.on_click(on_vacate_clicked)
t3_btn_students.on_click(on_view_students_clicked)
t3_btn_rooms.on_click(on_view_rooms_clicked)

# Render complete App Components
display(HTML("<h2>🌐 Cloud Hostel Management Portal</h2>"))
display(tab)
display(HTML("<hr><h4>📡 Live Cloud Server Output:</h4>"))
display(output_area)


Output()